# PSTU: Per-Secret-Type Unlearning

### Interactive Demo Notebook

---

**Paper:** *"Not All Secrets Are Equal: Type-Aware Unlearning for Language Model Secret Removal"*  
**Venue:** ECML-PKDD 2026

---

## The Problem

Large language models memorize sensitive data — SSH keys, API tokens, PII — and can reproduce them verbatim. Existing unlearning methods face a **stability-plasticity dilemma**:

| Method | Forgetting | Utility |
|--------|------------|---------|
| Gradient Ascent (GA) | Removes secrets | Destroys model |
| NPO / SimNPO | Stable | Leaves secrets intact at scale |
| **PSTU (Ours)** | Removes all secrets | Near-clean perplexity |

## The Solution: PSTU

PSTU is **training-free**. Given an infected model $\theta$ and a clean reference $\theta_0$:

1. Compute the **task vector**: $\tau = \theta - \theta_0$
2. Derive **per-type, per-layer-group** scaling from gradient saliency
3. (Optional) Apply **PSTU-Trim** to denoise at scale
4. Subtract adaptively: $\theta^* = \theta - \Lambda \odot \tau'$

> **This notebook** is a CPU-only demonstration of the core mechanics using random tensors.  
> For full experiments with real models, see the `scripts/` directory.

## Results at a Glance

![PSTU removes all memorized secrets while preserving model utility](../figures/fig_motivation.png)

**Key insight:** Different secret types (SSH keys vs. addresses vs. PINs) have different gradient-saliency footprints and require different correction strengths. A single global $\lambda$ cannot accommodate all types — PSTU uses **per-type, per-layer-group** scaling instead.

In [ ]:
# ============================================================
# Setup: create a tiny mock model for demonstration
# ============================================================

import torch
from pstu.method import apply_pstu, _compute_trim_threshold
from pstu.utils import detect_num_layers, param_group

torch.manual_seed(42)  # For reproducibility

# Simulate a 4-layer Pythia-style model (parameter names determine grouping)
PARAM_NAMES = [
    "gpt_neox.embed_in.weight",              # Embedding layer
    "gpt_neox.layers.0.attention.dense.weight",  # Layer 0: attention
    "gpt_neox.layers.1.mlp.dense_4h_to_h.weight", # Layer 1: MLP
    "gpt_neox.layers.2.attention.dense.weight",  # Layer 2: attention  
    "gpt_neox.layers.3.mlp.dense_4h_to_h.weight", # Layer 3: MLP
    "embed_out.weight",                       # Output head
]

# Create mock parameters
clean_model = {n: torch.zeros(8, 8) for n in PARAM_NAMES}          # theta_0: clean reference
infected_model = {n: torch.randn(8, 8) * 0.1 for n in PARAM_NAMES}  # theta: infected with secrets

# Task vector = infected - clean (captures what was learned during infection)
task_vectors = {n: infected_model[n] - clean_model[n] for n in PARAM_NAMES}

# Mock saliency scores (in practice, derived from gradient magnitudes)
saliency = {n: (i + 1) / len(PARAM_NAMES) for i, n in enumerate(PARAM_NAMES)}

# Detect architecture
n_layers = detect_num_layers(clean_model)
print(f"Detected {n_layers} transformer layers")

---

## Step 1: Parameter Grouping

PSTU partitions the model into **layer groups**, each receiving an independent correction strength $\alpha$:

| Group | Parameters |
|-------|-----------|
| `embed` | Input embeddings |
| `g0`, `g1`, ... | Transformer layer blocks |
| `head` | Output projection (LM head) |

This enables **fine-grained control**: high-entropy secrets (SSH keys) often reside in early layers, while semantic secrets (addresses) spread across later layers.

In [ ]:
# Show how each parameter maps to its group
print("Parameter Group Assignments (group_size=2):\n")
print(f"{'Group':>8}  {'Parameter Name'}")
print("-" * 50)
for n in PARAM_NAMES:
    group = param_group(n, n_layers, group_size=2)
    print(f"{group:>8}  {n}")

---

## Step 2: Adaptive Subtraction with PSTU-Trim

The core unlearning operation:
$$\theta^* = \theta - \Lambda \odot \tau'$$

where:
- $\Lambda$ combines base rate + saliency-weighted boost per group
- $\tau'$ is the (optionally trimmed) task vector

**PSTU-Trim** (for 7B+ models): Zeroes out low-magnitude task-vector entries, keeping only the outliers that encode secrets. This separates signal from noise at scale.

In [ ]:
# Per-group correction strengths (in practice, tuned via Optuna)
alphas = {"embed": 1.0, "head": 1.0, "g0": 1.0, "g1": 1.0}

print("=" * 60)
print("EXPERIMENT 1: Full task-vector subtraction (no trimming)")
print("=" * 60)

unlearned_full = apply_pstu(
    infected_model, clean_model, task_vectors, saliency, alphas,
    saliency_boost=0.0, n_layers=n_layers, group_size=2,
    trim_fraction=0.0, device="cpu"
)
max_gap = max((unlearned_full[n] - clean_model[n]).abs().max().item() for n in PARAM_NAMES)
print(f"Max |theta* - theta_0|: {max_gap:.2e}")
print("Result: Full subtraction perfectly recovers the clean model!\n")

print("=" * 60)
print("EXPERIMENT 2: PSTU-Trim (keep only top 50% of task vector)")
print("=" * 60)

trim_threshold = _compute_trim_threshold(task_vectors, 0.5, "cpu")
unlearned_trimmed = apply_pstu(
    infected_model, clean_model, task_vectors, saliency, alphas,
    saliency_boost=0.0, n_layers=n_layers, group_size=2,
    trim_fraction=0.5, device="cpu"
)
residual = sum((unlearned_trimmed[n] - clean_model[n]).abs().sum().item() for n in PARAM_NAMES)

print(f"Trim threshold (50th percentile): {trim_threshold:.4f}")
print(f"Residual task vector magnitude:   {residual:.4f}")
print("Result: Low-magnitude entries are preserved (utility retention)!")

---

## Visualization: Task Vector Magnitudes

Let's visualize how the task vector magnitude varies across layer groups — this is what PSTU exploits for targeted unlearning.

In [ ]:
import matplotlib.pyplot as plt

# Compute task vector magnitude per parameter
magnitudes = {n: task_vectors[n].abs().mean().item() for n in PARAM_NAMES}
groups = [param_group(n, n_layers, group_size=2) for n in PARAM_NAMES]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Task vector magnitude by parameter
ax1 = axes[0]
short_names = [n.split('.')[-1][:15] for n in PARAM_NAMES]
colors = ['#2ecc71' if g == 'embed' else '#e74c3c' if g == 'head' else '#3498db' for g in groups]
ax1.bar(range(len(PARAM_NAMES)), list(magnitudes.values()), color=colors)
ax1.set_xticks(range(len(PARAM_NAMES)))
ax1.set_xticklabels(short_names, rotation=45, ha='right')
ax1.set_ylabel('Mean |task vector|')
ax1.set_title('Task Vector Magnitude by Parameter')
ax1.axhline(y=trim_threshold, color='red', linestyle='--', label=f'Trim threshold ({trim_threshold:.3f})')
ax1.legend()

# Plot 2: Saliency scores
ax2 = axes[1]
ax2.bar(range(len(PARAM_NAMES)), list(saliency.values()), color=colors)
ax2.set_xticks(range(len(PARAM_NAMES)))
ax2.set_xticklabels(short_names, rotation=45, ha='right')
ax2.set_ylabel('Saliency score')
ax2.set_title('Mock Saliency by Parameter')

plt.tight_layout()
plt.show()
print("\nColors: Green=embed, Blue=transformer layers, Red=head")

---

## Running Full Experiments

This demo used mock tensors. For **real experiments** with actual models and metrics:

### Quick Start

```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Infect a base model with synthetic secrets
MODEL_SIZE=1.4b EPOCHS=4 python scripts/infect_model.py

# 3. Run PSTU with Optuna hyperparameter search
python scripts/run_pstu.py --model pythia-1.4b --n-trials 500

# 4. Evaluate the result
python scripts/evaluate_model.py \
    --model-path results/pstu_comprehensive/pythia-1.4b_best_model_final \
    --clean-model EleutherAI/pythia-1.4b
```

### Paper Results

| Model | Secrets Removed | PPL Overhead |
|-------|-----------------|--------------|
| Pythia-1.4B | 175/175 | +1.3% |
| Pythia-2.8B | 175/175 | +2.4% |
| Pythia-6.9B | 175/175 | +3.2% (with Trim) |
| Llama-3.1-8B | 175/175 | +14.4% (with Trim) |

---

**License:** Code is Apache-2.0 | Data is CC-BY-4.0 | See `data/DATACARD.md`